In [1]:
import sympy as sp
from IPython.display import Math, display

sp.init_printing()

t = sp.symbols("t", real=True)
I = sp.I
eps = sp.symbols("epsilon", real=True)

E_pi = sp.Function(r"\mathcal{E}_\pi", real=True)(t)
E_x_3 = sp.Function(r"\mathcal{E}_x^{(3)}", real=True)(t)
E_x_5 = sp.Function(r"\mathcal{E}_x^{(5)}", real=True)(t)
E_y_1 = sp.Function(r"\mathcal{E}_y^{(1)}", real=True)(t)
E_y_3 = sp.Function(r"\mathcal{E}_y^{(3)}", real=True)(t)
E_y_5 = sp.Function(r"\mathcal{E}_y^{(5)}", real=True)(t)
delta_1_2 = sp.Function(r"\delta_1^{(2)}", real=True)(t)
delta_1_4 = sp.Function(r"\delta_1^{(4)}", real=True)(t)

Delta = sp.Function(r"\Delta", real=True)(t)
lam = sp.Function(r"\lambda", real=True)(t)

E_x = eps * E_pi + eps**3 * E_x_3 + eps**5 * E_x_5
E_y = eps * E_y_1 + eps**3 * E_y_3 + eps**5 * E_y_5
delta_1 = eps**2 * delta_1_2 + eps**4 * delta_1_4
delta_2 = Delta + 2 * delta_1

def ketbra(j, k):
    M = sp.zeros(3)
    M[j, k] = 1
    return M

Pi_0 = ketbra(0, 0)
Pi_1 = ketbra(1, 1)
Pi_2 = ketbra(2, 2)

sigma_x_01 = ketbra(0, 1) + ketbra(1, 0)
sigma_x_12 = ketbra(1, 2) + ketbra(2, 1)
sigma_x_02 = ketbra(0, 2) + ketbra(2, 0)

sigma_y_01 = -I * ketbra(0, 1) + I * ketbra(1, 0)
sigma_y_12 = -I * ketbra(1, 2) + I * ketbra(2, 1)
sigma_y_02 = -I * ketbra(0, 2) + I * ketbra(2, 0)

K = sigma_y_01 + lam * sigma_y_12

H_eff = (
    delta_1 * Pi_1
    + delta_2 * Pi_2
    + E_x / 2 * (sigma_x_01 + lam * sigma_x_12)
    + E_y / 2 * (sigma_y_01 + lam * sigma_y_12)
)

# Konvention aus Gl. (9): [A,S]_n = [[A,S]_{n-1},S]
S_1 = I * E_x / (2 * Delta) * (
    sigma_y_01 + lam * sigma_y_12
)

S_2 = I * lam * E_x**2 / (8 * Delta**2) * sigma_y_02

S = S_1 + S_2
# S = I * E_x / (2 * Delta) * K

In [2]:
def commutator(A, B):
    return A * B - B * A

def nested_adjoint(A, S, n):
    result = A
    for _ in range(n):
        result = commutator(result, S)
    return result

def truncate_epsilon(expr, max_order):
    return sp.series(sp.expand(expr), eps, 0, max_order + 1).removeO()

def truncate_matrix_epsilon(M, max_order):
    return sp.Matrix(M).applyfunc(lambda expr: truncate_epsilon(expr, max_order))

def bch_terms(H, S, max_order):
    dS = sp.diff(S, t)
    hamiltonian_terms = [nested_adjoint(H, S, n) / sp.factorial(n) for n in range(max_order + 1)]
    inertial_terms = [-I * nested_adjoint(dS, S, n) / sp.factorial(n + 1) for n in range(max_order)]
    return hamiltonian_terms, inertial_terms

def transform_bch(H, S, max_order):
    hamiltonian_terms, inertial_terms = bch_terms(H, S, max_order)
    return sum(hamiltonian_terms, sp.zeros(3)) + sum(inertial_terms, sp.zeros(3))

def set_ground_energy_zero(H):
    H = sp.Matrix(H)
    return H - H[0, 0] * sp.eye(3)

def operator_coefficients(H):
    H = set_ground_energy_zero(H)
    return {
        r"\Pi_0": H[0, 0],
        r"\Pi_1": H[1, 1],
        r"\Pi_2": H[2, 2],
        r"\sigma_{0,1}^{x}": (H[0, 1] + H[1, 0]) / 2,
        r"\sigma_{0,1}^{y}": (H[1, 0] - H[0, 1]) / (2 * I),
        r"\sigma_{1,2}^{x}": (H[1, 2] + H[2, 1]) / 2,
        r"\sigma_{1,2}^{y}": (H[2, 1] - H[1, 2]) / (2 * I),
        r"\sigma_{0,2}^{x}": (H[0, 2] + H[2, 0]) / 2,
        r"\sigma_{0,2}^{y}": (H[2, 0] - H[0, 2]) / (2 * I),
    }

def coefficient_at_order(expr, order):
    return sp.expand(truncate_epsilon(expr, order)).coeff(eps, order)

def coefficients_upto_order(coefficients, order):
    return {
        operator: clean_expression(truncate_epsilon(coefficient, order))
        for operator, coefficient in coefficients.items()
    }

def clean_expression(expr):
    expr = sp.cancel(sp.together(expr))
    numerator, denominator = sp.fraction(expr)
    numerator = sp.factor_terms(sp.factor(numerator))
    denominator = sp.factor(denominator)
    return sp.collect(sp.cancel(numerator / denominator), [eps, E_pi], exact=False)

def substitute_solution(expr, variable, solution, max_derivative=5):
    result = expr
    for order in range(max_derivative, 0, -1):
        result = result.subs(sp.diff(variable, t, order), sp.diff(solution, t, order))
    return result.subs(variable, solution).doit()

def apply_solutions(expr, solutions, max_derivative=5):
    result = expr
    for variable, solution in solutions.items():
        result = substitute_solution(result, variable, solution, max_derivative)
    return result

def solve_unique(expr, variable):
    expr = clean_expression(expr)
    solutions = sp.solve(sp.Eq(expr, 0), variable, simplify=False)

    if len(solutions) != 1:
        raise ValueError(f"Expected one solution for {variable}, obtained {len(solutions)}.")

    return clean_expression(solutions[0])

def display_solution(variable, solution):
    display(Math(sp.latex(variable) + "=" + sp.latex(clean_expression(solution))))


In [ ]:
check_S2 = coefficient_at_order(coefficients[r"\sigma_{0,2}^{x}"], 2)
check_S2 = apply_solutions(check_S2, solutions)
check_S2 = clean_expression(check_S2)

display(Math(
    r"c_{\sigma_{0,2}^{x}}^{(2)}="
    + sp.latex(check_S2)
))

In [ ]:
max_order = 5

H_V_bch = transform_bch(H_eff, S, max_order)
H_V_bch = truncate_matrix_epsilon(H_V_bch, max_order)
coefficients = operator_coefficients(H_V_bch)

key_x = r"\sigma_{0,1}^{x}"
key_y = r"\sigma_{0,1}^{y}"
key_d = r"\Pi_1"

solutions = {}

# Ordnung 1: Quadraturkorrektur
eq_y_1 = coefficient_at_order(coefficients[key_y], 1)
solutions[E_y_1] = solve_unique(eq_y_1, E_y_1)
print("h")
# Ordnung 2: Detuningkorrektur
eq_d_2 = coefficient_at_order(coefficients[key_d], 2)
eq_d_2 = apply_solutions(eq_d_2, solutions)
solutions[delta_1_2] = solve_unique(eq_d_2, delta_1_2)

# Ordnung 3: In-phase- und Quadraturkorrektur
eq_x_3 = coefficient_at_order(coefficients[key_x], 3)
eq_x_3 = apply_solutions(eq_x_3, solutions)
solutions[E_x_3] = solve_unique(eq_x_3, E_x_3)

eq_y_3 = coefficient_at_order(coefficients[key_y], 3)
eq_y_3 = apply_solutions(eq_y_3, solutions)
solutions[E_y_3] = solve_unique(eq_y_3, E_y_3)

# Ordnung 4: Detuningkorrektur
eq_d_4 = coefficient_at_order(coefficients[key_d], 4)
eq_d_4 = apply_solutions(eq_d_4, solutions)
solutions[delta_1_4] = solve_unique(eq_d_4, delta_1_4)

# Ordnung 5: In-phase- und Quadraturkorrektur
eq_x_5 = coefficient_at_order(coefficients[key_x], 5)
eq_x_5 = apply_solutions(eq_x_5, solutions)
solutions[E_x_5] = solve_unique(eq_x_5, E_x_5)

eq_y_5 = coefficient_at_order(coefficients[key_y], 5)
eq_y_5 = apply_solutions(eq_y_5, solutions)
solutions[E_y_5] = solve_unique(eq_y_5, E_y_5)

for variable, solution in solutions.items():
    display_solution(variable, solution)

In [ ]:
def paper_latex(expr):
    latex = sp.latex(clean_expression(expr))

    replacements = [
        (sp.latex(sp.diff(E_pi, t)), r"\dot{\mathcal{E}}_\pi(t)"),
        (sp.latex(sp.diff(Delta, t)), r"\dot{\Delta}(t)"),
        (sp.latex(sp.diff(lam, t)), r"\dot{\lambda}(t)"),
        (sp.latex(E_pi), r"\mathcal{E}_\pi(t)"),
        (sp.latex(Delta), r"\Delta(t)"),
        (sp.latex(lam), r"\lambda(t)"),
    ]

    for old, new in replacements:
        latex = latex.replace(old, new)

    return latex

def latex_order_terms(expr, orders):
    terms = []

    for order in orders:
        coefficient = clean_expression(coefficient_at_order(expr, order))

        if coefficient == 0:
            continue

        negative = coefficient.could_extract_minus_sign()

        if negative:
            coefficient = -coefficient

        terms.append((negative, paper_latex(coefficient)))

    return terms

def latex_drag_equation(lhs, expr, orders, label, punctuation=","):
    terms = latex_order_terms(expr, orders)

    if not terms:
        return lhs + "\n&=0" + punctuation + rf" \label{{{label}}}"

    negative, term = terms[0]
    rows = [lhs + "\n&= " + ("- " if negative else "") + term]

    for negative, term in terms[1:]:
        rows[-1] += r" \nonumber \\"
        rows.append(r"&\quad " + ("- " if negative else "+ ") + term)

    rows[-1] += punctuation + rf" \label{{{label}}}"
    return "\n".join(rows)


E_x_solution = truncate_epsilon(apply_solutions(E_x, solutions), max_order)
E_y_solution = truncate_epsilon(apply_solutions(E_y, solutions), max_order)
delta_1_solution = truncate_epsilon(apply_solutions(delta_1, solutions), max_order)

latex_code = (
    r"\begin{subequations}" + "\n"
    r"\label{eq:drag_controls}" + "\n"
    r"\begin{align}" + "\n"
    + latex_drag_equation(
        r"\mathcal{E}^{x}(t)",
        E_x_solution,
        [1, 3, 5],
        "eq:drag_controls_a"
    )
    + r" \\[0.6em]" + "\n"
    + latex_drag_equation(
        r"\mathcal{E}^{y}(t)",
        E_y_solution,
        [1, 3, 5],
        "eq:drag_controls_b"
    )
    + r" \\[0.6em]" + "\n"
    + latex_drag_equation(
        r"\delta_1(t)",
        delta_1_solution,
        [2, 4],
        "eq:drag_controls_c",
        punctuation="."
    )
    + "\n"
    r"\end{align}" + "\n"
    r"\end{subequations}"
)

import re

rendered_latex = latex_code
rendered_latex = rendered_latex.replace(r"\begin{subequations}", "")
rendered_latex = rendered_latex.replace(r"\end{subequations}", "")
rendered_latex = rendered_latex.replace(r"\begin{align}", r"\begin{aligned}")
rendered_latex = rendered_latex.replace(r"\end{align}", r"\end{aligned}")
rendered_latex = rendered_latex.replace(r"\nonumber", "")
rendered_latex = re.sub(r"\\label\{[^}]+\}", "", rendered_latex)

display(Math(rendered_latex))
print(latex_code)

<IPython.core.display.Math object>

\begin{subequations}
\label{eq:drag_controls}
\begin{align}
\mathcal{E}^{x}(t)
&= \mathcal{E}_\pi(t) \nonumber \\
&\quad + \frac{\left(\lambda^{2}{\left(t \right)} - 4\right) \mathcal{E}_\pi^{3}{\left(t \right)}}{8 \Delta^{2}{\left(t \right)}} \nonumber \\
&\quad + \frac{\left(- 11 \lambda^{4}{\left(t \right)} - 55 \lambda^{2}{\left(t \right)} + 208\right) \mathcal{E}_\pi^{5}{\left(t \right)}}{384 \Delta^{4}{\left(t \right)}}, \label{eq:drag_controls_a} \\[0.6em]
\mathcal{E}^{y}(t)
&= \frac{- \Delta(t) \dot{\mathcal{E}}_\pi(t) + \mathcal{E}_\pi(t) \dot{\Delta}(t)}{\Delta^{2}{\left(t \right)}} \nonumber \\
&\quad + \frac{\left(- 9 \Delta(t) \lambda^{2}{\left(t \right)} \mathcal{E}_\pi^{2}{\left(t \right)} + 36 \Delta(t) \mathcal{E}_\pi^{2}{\left(t \right)}\right) \dot{\mathcal{E}}_\pi(t) + \left(- 7 \Delta(t) \lambda(t) \dot{\lambda}(t) + 9 \lambda^{2}{\left(t \right)} \dot{\Delta}(t) - 36 \dot{\Delta}(t)\right) \mathcal{E}_\pi^{3}{\left(t \right)}}{24 \Delta^{4}{\left(t \right)}} \nonu

In [ ]:
check_S2 = coefficient_at_order(coefficients[r"\sigma_{0,2}^{x}"], 2)
check_S2 = apply_solutions(check_S2, solutions)
check_S2 = clean_expression(check_S2)

display(Math(
    r"c_{\sigma_{0,2}^{x}}^{(2)}="
    + sp.latex(check_S2)
))

<IPython.core.display.Math object>